# Reading a PDF

From bytes to clean page text: validation, metadata, running headers and footers, and line wrapping.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

## A synthetic book
The tests build PDFs with known structure (fpdf2). Copyrighted books are never committed; put your own in `test-books/` to try one.

In [ ]:
from tests.pdfs import make_book, chapter, part
data = make_book([part('PART ONE', outline='Part One'),
                  chapter('CHAPTER I', 'the river', outline='Chapter I', level=1),
                  chapter('CHAPTER II', 'the axe', outline='Chapter II', level=1)],
                 title='The Demo Book', author='A. Writer')
len(data), data[:8]

## `read_pdf`: validate and extract

In [ ]:
from marginalia.ingest.pdf import read_pdf, PdfError
doc = read_pdf(data)
print('pages:', len(doc.pages), '| title:', doc.title, '| author:', doc.author)
print('outline:', doc.outline)
print('sha256:', doc.sha256[:16], '... (the book id is built from it)')
print(doc.pages[1][:400])

## What is refused, and why
Every refusal has a stable code the API and the UI show.

In [ ]:
from tests.pdfs import scanned_book
for label, bad in [('not a PDF', b'PK\x03\x04 an epub, really'),
                   ('encrypted', make_book([chapter('CHAPTER I', 'the river')], encrypt=('user', 'owner'))),
                   ('a scan, no text layer', scanned_book())]:
    try:
        read_pdf(bad)
    except PdfError as e:
        print(f'{label:24} -> {e.code}: {e}')

## Running headers and footers
Lines repeated at the top or bottom of many pages (the book title, page numbers) are found by repetition and removed before chapter detection.

In [ ]:
from marginalia.ingest import layout
pages = [layout.normalize_page(p) for p in doc.pages]
running = layout.find_running_lines(pages)
print('running lines:', running)
clean = [layout.strip_running_lines(p, running) for p in pages]
print(clean[1][:300])

## Line wrapping
PDF text breaks lines where the page did. The book's wrap style decides how lines are joined back into paragraphs.

In [ ]:
text = '\n'.join(clean)
style = layout.detect_wrap_style(text)
print('wrap style:', style)
paragraphs = layout.to_paragraphs(text, style, layout.word_counts(text))
print(paragraphs[:500])

## Your own book (optional)

In [ ]:
from pathlib import Path
own = sorted(Path('test-books').glob('*.pdf')) + sorted(Path('library').glob('*/source.pdf'))
if own:
    d = read_pdf(own[0])
    print(own[0], '|', len(d.pages), 'pages |', d.title, '|', d.author)
    print(d.pages[min(5, len(d.pages) - 1)][:600])
else:
    print('no PDF in test-books/ or library/')